# Exploración de features

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parents[0]))

from scripts.data.load_data import cargar_raw
from scripts.data.clean_data import limpiar_datos

df = cargar_raw("titanic_dataset.csv")

df_limpio = limpiar_datos(df)

df_exploratorio = df_limpio.copy()

In [2]:
df_exploratorio.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,7.2500,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,71.2833,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,7.9250,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,53.1000,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,8.0500,S


In [3]:
import pandas as pd
import numpy as np

# modelado
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# métricas
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

# encoding
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score



In [4]:
def evaluar_features(df, features):
    
    X = df[features]
    y = df["Survived"]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    modelo = LogisticRegression(max_iter=1000)
    modelo.fit(X_train, y_train)
    
    y_pred = modelo.predict(X_test)
    y_prob = modelo.predict_proba(X_test)[:, 1]
    
    return {
        "features": features,
        "f1": f1_score(y_test, y_pred),
        "auc": roc_auc_score(y_test, y_prob)
    }

## SibSp y Parch

In [5]:
evaluar_features(df_exploratorio, ["SibSp", "Parch"])

{'features': ['SibSp', 'Parch'],
 'f1': 0.16091954022988506,
 'auc': 0.5287644787644787}

In [6]:
df_exploratorio['FamilySize'] = df_exploratorio['SibSp'] + df_exploratorio['Parch'] + 1

In [7]:
evaluar_features(df_exploratorio, ["FamilySize"])

{'features': ['FamilySize'], 'f1': 0.0, 'auc': 0.624002574002574}

In [8]:
evaluar_features(df_exploratorio, ["FamilySize", "Age", "Fare"])

{'features': ['FamilySize', 'Age', 'Fare'],
 'f1': 0.3541666666666667,
 'auc': 0.7936293436293437}

In [9]:
evaluar_features(df_exploratorio, ["SibSp","Parch", "Age", "Fare"])

{'features': ['SibSp', 'Parch', 'Age', 'Fare'],
 'f1': 0.44660194174757284,
 'auc': 0.7655727155727156}

In [10]:
def FamilySize_agrupado (size):
    if size == 1:
        return "Solo"
    elif 2 <= size <= 4:
        return "Pequeña"
    else:
        return "Grande"

In [11]:
df_exploratorio['FamilySize_agrupado'] = df_exploratorio['FamilySize'].apply(FamilySize_agrupado)
df_exploratorio["FamilySize_agrupado"] = pd.Categorical(df_exploratorio["FamilySize_agrupado"], categories=["Solo", "Pequeña", "Grande"])

In [12]:
df_exploratorio = pd.get_dummies( df_exploratorio, columns=["FamilySize_agrupado"], drop_first=True, dtype=int)

In [13]:
df_exploratorio.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Embarked,FamilySize,FamilySize_agrupado_Pequeña,FamilySize_agrupado_Grande
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,7.2500,S,2,1,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,71.2833,C,2,1,0
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,7.9250,S,1,0,0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,53.1000,S,2,1,0
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,8.0500,S,1,0,0


In [14]:
evaluar_features(df_exploratorio, ["FamilySize_agrupado_Pequeña", "FamilySize_agrupado_Grande", "Age", "Fare"])

{'features': ['FamilySize_agrupado_Pequeña',
  'FamilySize_agrupado_Grande',
  'Age',
  'Fare'],
 'f1': 0.5641025641025641,
 'auc': 0.7941441441441442}

In [15]:
evaluar_features(df_exploratorio, ['FamilySize_agrupado_Pequeña', 'FamilySize_agrupado_Grande'])

{'features': ['FamilySize_agrupado_Pequeña', 'FamilySize_agrupado_Grande'],
 'f1': 0.5970149253731343,
 'auc': 0.6837837837837839}

## Name

In [16]:
df_exploratorio['Name'].shape

(891,)

In [17]:
df_exploratorio["Title"] = df_exploratorio["Name"].str.extract(r",\s*([^\.]+)\.")
df_exploratorio["Title"].value_counts()

Title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Mlle              2
Major             2
Col               2
the Countess      1
Capt              1
Ms                1
Sir               1
Lady              1
Mme               1
Don               1
Jonkheer          1
Name: count, dtype: int64

In [18]:
def Title_agrupado(t):
    if t in ["Mr"]:
        return "Hombre"
    elif t in ["Mrs", "Miss"]:
        return "Mujer"
    elif t in ["Master"]:
        return "Niño"
    else:
        return "Rare"

In [19]:
df_exploratorio["Title_agrupado"] = df_exploratorio["Title"].apply(Title_agrupado)

In [20]:
df_exploratorio["Title_agrupado"] = pd.Categorical(
    df_exploratorio["Title_agrupado"],
    categories=["Hombre", "Mujer", "Niño", "Rare"]
)

In [21]:
df_exploratorio = pd.get_dummies(
    df_exploratorio,
    columns=["Title_agrupado"],
    drop_first=True, 
    dtype=int
)

In [22]:
df_exploratorio.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Embarked,FamilySize,FamilySize_agrupado_Pequeña,FamilySize_agrupado_Grande,Title,Title_agrupado_Mujer,Title_agrupado_Niño,Title_agrupado_Rare
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,7.2500,S,2,1,0,Mr,0,0,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,71.2833,C,2,1,0,Mrs,1,0,0
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,7.9250,S,1,0,0,Miss,1,0,0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,53.1000,S,2,1,0,Mrs,1,0,0
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,8.0500,S,1,0,0,Mr,0,0,0


In [23]:
evaluar_features(df_exploratorio, ["Title_agrupado_Rare", "Title_agrupado_Mujer", "Title_agrupado_Niño"])

{'features': ['Title_agrupado_Rare',
  'Title_agrupado_Mujer',
  'Title_agrupado_Niño'],
 'f1': 0.7183098591549296,
 'auc': 0.7954311454311455}

In [24]:
evaluar_features(df_exploratorio, ["Title_agrupado_Rare", "Title_agrupado_Mujer", "Title_agrupado_Niño", "Age", "Fare"])

{'features': ['Title_agrupado_Rare',
  'Title_agrupado_Mujer',
  'Title_agrupado_Niño',
  'Age',
  'Fare'],
 'f1': 0.7222222222222223,
 'auc': 0.8473616473616473}

In [25]:
evaluar_features(df_exploratorio, ["Title_agrupado_Rare", 
                                   "Title_agrupado_Mujer",
                                    "Title_agrupado_Niño", 
                                    "Age", 
                                    "Fare", 
                                    "FamilySize_agrupado_Pequeña", 
                                    "FamilySize_agrupado_Grande" ])

{'features': ['Title_agrupado_Rare',
  'Title_agrupado_Mujer',
  'Title_agrupado_Niño',
  'Age',
  'Fare',
  'FamilySize_agrupado_Pequeña',
  'FamilySize_agrupado_Grande'],
 'f1': 0.7692307692307693,
 'auc': 0.8848133848133848}

## Fare

In [26]:
df_exploratorio['Fare_log'] = np.log1p(df_exploratorio['Fare'])

In [27]:
df_exploratorio['Fare'].describe()

count    891.000000
mean      32.204208
std       49.693429
min        0.000000
25%        7.910400
50%       14.454200
75%       31.000000
max      512.329200
Name: Fare, dtype: float64

In [28]:
df_exploratorio['Fare_log'].describe()

count    891.000000
mean       2.962246
std        0.969048
min        0.000000
25%        2.187218
50%        2.737881
75%        3.465736
max        6.240917
Name: Fare_log, dtype: float64

In [29]:
evaluar_features(df_exploratorio, ["Fare"])

{'features': ['Fare'], 'f1': 0.3541666666666667, 'auc': 0.7490347490347491}

In [30]:
evaluar_features(df_exploratorio, ["Fare_log"])

{'features': ['Fare_log'],
 'f1': 0.47706422018348627,
 'auc': 0.7490347490347491}

In [31]:
evaluar_features(df_exploratorio, ["Title_agrupado_Rare", 
                                   "Title_agrupado_Mujer",
                                    "Title_agrupado_Niño", 
                                    "Age", 
                                    "Fare_log", 
                                    "FamilySize_agrupado_Pequeña", 
                                    "FamilySize_agrupado_Grande" ])

{'features': ['Title_agrupado_Rare',
  'Title_agrupado_Mujer',
  'Title_agrupado_Niño',
  'Age',
  'Fare_log',
  'FamilySize_agrupado_Pequeña',
  'FamilySize_agrupado_Grande'],
 'f1': 0.7692307692307693,
 'auc': 0.8966537966537966}

Hay variables más fuertes que están “absorbiendo” el efecto de Fare

In [32]:
df_exploratorio.groupby(pd.qcut(df["Fare"], 4))["Survived"].mean()

Fare
(-0.001, 7.91]     0.197309
(7.91, 14.454]     0.303571
(14.454, 31.0]     0.454955
(31.0, 512.329]    0.581081
Name: Survived, dtype: float64

## Age

In [33]:
df_exploratorio.groupby(pd.qcut(df["Age"], 5))["Survived"].mean()

Age
(0.419, 19.0]    0.481707
(19.0, 25.0]     0.328467
(25.0, 31.8]     0.393701
(31.8, 41.0]     0.437500
(41.0, 80.0]     0.373239
Name: Survived, dtype: float64

In [34]:
evaluar_features(df_exploratorio, ["Age"])

{'features': ['Age'], 'f1': 0.0, 'auc': 0.5488416988416989}

In [35]:
def Age_agrupado(age):
    if age < 12:
        return "Niño"
    elif 12 <= age < 18:
        return "Adolescente"
    elif 18 <= age < 35:
        return "Joven"
    elif 35 <= age < 60:
        return "Adulto"
    else:
        return "Mayor"

In [36]:
df_exploratorio['Age_bin'] = df_exploratorio['Age'].apply(Age_agrupado)

In [37]:
df_exploratorio["Age_bin"] = pd.Categorical(
    df_exploratorio["Age_bin"],
    categories=["Adulto", "Joven", "Niño", "Adolescente", "Mayor"]
)



In [38]:
df_exploratorio = pd.get_dummies(
    df_exploratorio,
    columns=["Age_bin"],
    drop_first=True,
    dtype=int
)

In [39]:
df_exploratorio.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Embarked,...,FamilySize_agrupado_Grande,Title,Title_agrupado_Mujer,Title_agrupado_Niño,Title_agrupado_Rare,Fare_log,Age_bin_Joven,Age_bin_Niño,Age_bin_Adolescente,Age_bin_Mayor
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,7.2500,S,...,0,Mr,0,0,0,2.110213,1,0,0,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,71.2833,C,...,0,Mrs,1,0,0,4.280593,0,0,0,0
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,7.9250,S,...,0,Miss,1,0,0,2.188856,1,0,0,0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,53.1000,S,...,0,Mrs,1,0,0,3.990834,0,0,0,0
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,8.0500,S,...,0,Mr,0,0,0,2.202765,0,0,0,0


In [40]:
evaluar_features(df_exploratorio, ["Age_bin_Joven", "Age_bin_Niño", "Age_bin_Adolescente", "Age_bin_Mayor"])

{'features': ['Age_bin_Joven',
  'Age_bin_Niño',
  'Age_bin_Adolescente',
  'Age_bin_Mayor'],
 'f1': 0.16470588235294117,
 'auc': 0.5465894465894466}

In [41]:
evaluar_features(df_exploratorio, ["Age_bin_Joven", 
                                   "Age_bin_Niño", 
                                   "Age_bin_Adolescente", 
                                   "Age_bin_Mayor",
                                   "Title_agrupado_Rare", 
                                   "Title_agrupado_Mujer",
                                    "Title_agrupado_Niño",  
                                    "Fare_log", 
                                    "FamilySize_agrupado_Pequeña", 
                                    "FamilySize_agrupado_Grande"])

{'features': ['Age_bin_Joven',
  'Age_bin_Niño',
  'Age_bin_Adolescente',
  'Age_bin_Mayor',
  'Title_agrupado_Rare',
  'Title_agrupado_Mujer',
  'Title_agrupado_Niño',
  'Fare_log',
  'FamilySize_agrupado_Pequeña',
  'FamilySize_agrupado_Grande'],
 'f1': 0.7692307692307693,
 'auc': 0.8931788931788931}

In [42]:
evaluar_features(df_exploratorio, ["Age_bin_Joven", 
                                   "Age_bin_Niño", 
                                   "Age_bin_Adolescente", 
                                   "Age_bin_Mayor", 
                                    "Fare_log", 
                                    "FamilySize_agrupado_Pequeña", 
                                    "FamilySize_agrupado_Grande"])

{'features': ['Age_bin_Joven',
  'Age_bin_Niño',
  'Age_bin_Adolescente',
  'Age_bin_Mayor',
  'Fare_log',
  'FamilySize_agrupado_Pequeña',
  'FamilySize_agrupado_Grande'],
 'f1': 0.6115702479338844,
 'auc': 0.8037966537966538}

In [43]:
evaluar_features(df_exploratorio, ["Title_agrupado_Rare", 
                                   "Title_agrupado_Mujer",
                                    "Title_agrupado_Niño",  
                                    "Fare_log", 
                                    "FamilySize_agrupado_Pequeña", 
                                    "FamilySize_agrupado_Grande"])

{'features': ['Title_agrupado_Rare',
  'Title_agrupado_Mujer',
  'Title_agrupado_Niño',
  'Fare_log',
  'FamilySize_agrupado_Pequeña',
  'FamilySize_agrupado_Grande'],
 'f1': 0.7692307692307693,
 'auc': 0.8945945945945946}

## Sex y Embarked

In [44]:
df_exploratorio = pd.get_dummies(
    df_exploratorio,
    columns=["Sex", "Embarked"],  
    drop_first=True,
    dtype=int
)

In [45]:
evaluar_features(df_exploratorio, ["Sex_male"])

{'features': ['Sex_male'], 'f1': 0.7272727272727273, 'auc': 0.7703989703989704}

In [46]:
evaluar_features(df_exploratorio, ["Title_agrupado_Rare", 
                                   "Title_agrupado_Mujer",
                                    "Title_agrupado_Niño",  
                                    "Fare_log", 
                                    "Sex_male",
                                    "FamilySize_agrupado_Pequeña", 
                                    "FamilySize_agrupado_Grande"])

{'features': ['Title_agrupado_Rare',
  'Title_agrupado_Mujer',
  'Title_agrupado_Niño',
  'Fare_log',
  'Sex_male',
  'FamilySize_agrupado_Pequeña',
  'FamilySize_agrupado_Grande'],
 'f1': 0.7777777777777778,
 'auc': 0.8974259974259974}

In [47]:
evaluar_features(df_exploratorio, ["Embarked_Q", "Embarked_S"])

{'features': ['Embarked_Q', 'Embarked_S'],
 'f1': 0.4273504273504274,
 'auc': 0.5996138996138997}

In [48]:
evaluar_features(df_exploratorio, ["Title_agrupado_Rare", 
                                   "Title_agrupado_Mujer",
                                    "Title_agrupado_Niño",  
                                    "Fare_log", 
                                    "Sex_male",
                                    "Embarked_Q", "Embarked_S",
                                    "FamilySize_agrupado_Pequeña", 
                                    "FamilySize_agrupado_Grande"])

{'features': ['Title_agrupado_Rare',
  'Title_agrupado_Mujer',
  'Title_agrupado_Niño',
  'Fare_log',
  'Sex_male',
  'Embarked_Q',
  'Embarked_S',
  'FamilySize_agrupado_Pequeña',
  'FamilySize_agrupado_Grande'],
 'f1': 0.7777777777777778,
 'auc': 0.8938867438867439}

In [49]:
evaluar_features(df_exploratorio, ["Pclass"])

{'features': ['Pclass'], 'f1': 0.5826771653543307, 'auc': 0.7087516087516088}

In [50]:
evaluar_features(df_exploratorio, ["Title_agrupado_Rare", 
                                   "Title_agrupado_Mujer",
                                    "Title_agrupado_Niño",  
                                    "Fare_log", 
                                    "Sex_male",
                                    "Pclass",
                                    "Embarked_Q", "Embarked_S",
                                    "FamilySize_agrupado_Pequeña", 
                                    "FamilySize_agrupado_Grande"])

{'features': ['Title_agrupado_Rare',
  'Title_agrupado_Mujer',
  'Title_agrupado_Niño',
  'Fare_log',
  'Sex_male',
  'Pclass',
  'Embarked_Q',
  'Embarked_S',
  'FamilySize_agrupado_Pequeña',
  'FamilySize_agrupado_Grande'],
 'f1': 0.7832167832167832,
 'auc': 0.8969755469755469}

In [51]:
pclass_dummies = pd.get_dummies(
    df_exploratorio["Pclass"],
    prefix="Pclass",
    drop_first=True,
    dtype=int
)

df_exploratorio = pd.concat([df_exploratorio, pclass_dummies], axis=1)

In [52]:
evaluar_features(df_exploratorio, ["Pclass_2", "Pclass_3"])

{'features': ['Pclass_2', 'Pclass_3'],
 'f1': 0.5826771653543307,
 'auc': 0.7087516087516088}

In [53]:
evaluar_features(df_exploratorio, ["Title_agrupado_Rare", 
                                   "Title_agrupado_Mujer",
                                    "Title_agrupado_Niño",  
                                    "Fare_log", 
                                    "Sex_male",
                                    "Pclass_2", "Pclass_3",
                                    "Embarked_Q", "Embarked_S",
                                    "FamilySize_agrupado_Pequeña", 
                                    "FamilySize_agrupado_Grande"])

{'features': ['Title_agrupado_Rare',
  'Title_agrupado_Mujer',
  'Title_agrupado_Niño',
  'Fare_log',
  'Sex_male',
  'Pclass_2',
  'Pclass_3',
  'Embarked_Q',
  'Embarked_S',
  'FamilySize_agrupado_Pequeña',
  'FamilySize_agrupado_Grande'],
 'f1': 0.7832167832167832,
 'auc': 0.8963320463320463}

## Interacciones

In [54]:
evaluar_features(df_exploratorio, ["Title_agrupado_Rare", 
                                   "Title_agrupado_Mujer",
                                    "Title_agrupado_Niño",   
                                    "Sex_male"])

{'features': ['Title_agrupado_Rare',
  'Title_agrupado_Mujer',
  'Title_agrupado_Niño',
  'Sex_male'],
 'f1': 0.7272727272727273,
 'auc': 0.7981338481338481}

In [55]:
evaluar_features(df_exploratorio, ["Title_agrupado_Rare", 
                                   "Title_agrupado_Mujer",
                                    "Title_agrupado_Niño",   
                                    "Sex_male","Pclass",
                                    "Fare_log"])

{'features': ['Title_agrupado_Rare',
  'Title_agrupado_Mujer',
  'Title_agrupado_Niño',
  'Sex_male',
  'Pclass',
  'Fare_log'],
 'f1': 0.7448275862068966,
 'auc': 0.8787001287001287}

In [56]:
evaluar_features(df_exploratorio, ["Title_agrupado_Rare", 
                                   "Title_agrupado_Mujer",
                                    "Title_agrupado_Niño",   
                                    "Sex_male",#"Pclass",
                                    "Fare_log"])

{'features': ['Title_agrupado_Rare',
  'Title_agrupado_Mujer',
  'Title_agrupado_Niño',
  'Sex_male',
  'Fare_log'],
 'f1': 0.7346938775510203,
 'auc': 0.8675032175032176}

In [57]:
evaluar_features(df_exploratorio, ["Title_agrupado_Rare", 
                                   "Title_agrupado_Mujer",
                                    "Title_agrupado_Niño",   
                                    "Sex_male","Pclass",
                                ])

{'features': ['Title_agrupado_Rare',
  'Title_agrupado_Mujer',
  'Title_agrupado_Niño',
  'Sex_male',
  'Pclass'],
 'f1': 0.7448275862068966,
 'auc': 0.8796010296010297}

In [58]:
evaluar_features(df_exploratorio, ["Title_agrupado_Rare", 
                                   "Title_agrupado_Mujer",
                                    "Title_agrupado_Niño",   
                                    "Sex_male","Pclass",
                                    "Fare"])

{'features': ['Title_agrupado_Rare',
  'Title_agrupado_Mujer',
  'Title_agrupado_Niño',
  'Sex_male',
  'Pclass',
  'Fare'],
 'f1': 0.7448275862068966,
 'auc': 0.87998712998713}

In [59]:
evaluar_features(df_exploratorio, ["Fare_log", "Pclass"])

{'features': ['Fare_log', 'Pclass'],
 'f1': 0.5781250000000001,
 'auc': 0.7594594594594595}

In [60]:
evaluar_features(df_exploratorio, ["Fare_log", "Pclass" ])

{'features': ['Fare_log', 'Pclass'],
 'f1': 0.5781250000000001,
 'auc': 0.7594594594594595}

In [61]:
evaluar_features(df_exploratorio, ["Title_agrupado_Rare", 
                                   "Title_agrupado_Mujer",
                                    "Title_agrupado_Niño",   
                                    "Sex_male","Pclass",
                                    "Embarked_Q", "Embarked_S"])

{'features': ['Title_agrupado_Rare',
  'Title_agrupado_Mujer',
  'Title_agrupado_Niño',
  'Sex_male',
  'Pclass',
  'Embarked_Q',
  'Embarked_S'],
 'f1': 0.7625,
 'auc': 0.877992277992278}

In [62]:
evaluar_features(df_exploratorio, ["Title_agrupado_Rare", 
                                   "Title_agrupado_Mujer",
                                    "Title_agrupado_Niño",   
                                    "Sex_male","Pclass",
                                    "Embarked_Q", "Embarked_S",
                                    "FamilySize_agrupado_Pequeña",
                                    "FamilySize_agrupado_Grande"])

{'features': ['Title_agrupado_Rare',
  'Title_agrupado_Mujer',
  'Title_agrupado_Niño',
  'Sex_male',
  'Pclass',
  'Embarked_Q',
  'Embarked_S',
  'FamilySize_agrupado_Pequeña',
  'FamilySize_agrupado_Grande'],
 'f1': 0.7916666666666666,
 'auc': 0.8894465894465895}

In [63]:
evaluar_features(df_exploratorio, ["Title_agrupado_Rare", 
                                   "Title_agrupado_Mujer",
                                    "Title_agrupado_Niño",   
                                    "Sex_male","Pclass",
                                    #"Embarked_Q", "Embarked_S",
                                    "FamilySize_agrupado_Pequeña",
                                    "FamilySize_agrupado_Grande"])

{'features': ['Title_agrupado_Rare',
  'Title_agrupado_Mujer',
  'Title_agrupado_Niño',
  'Sex_male',
  'Pclass',
  'FamilySize_agrupado_Pequeña',
  'FamilySize_agrupado_Grande'],
 'f1': 0.7832167832167832,
 'auc': 0.8881595881595882}

In [64]:
evaluar_features(df_exploratorio, ["Title_agrupado_Rare", 
                                   "Title_agrupado_Mujer",
                                    "Title_agrupado_Niño",   
                                    "Sex_male","Pclass",
                                    "Embarked_Q", "Embarked_S",
                                    "FamilySize_agrupado_Pequeña",
                                    "FamilySize_agrupado_Grande",
                                    "Age_bin_Joven",
                                    "Age_bin_Niño",
                                    "Age_bin_Adolescente",
                                    "Age_bin_Mayor"])

{'features': ['Title_agrupado_Rare',
  'Title_agrupado_Mujer',
  'Title_agrupado_Niño',
  'Sex_male',
  'Pclass',
  'Embarked_Q',
  'Embarked_S',
  'FamilySize_agrupado_Pequeña',
  'FamilySize_agrupado_Grande',
  'Age_bin_Joven',
  'Age_bin_Niño',
  'Age_bin_Adolescente',
  'Age_bin_Mayor'],
 'f1': 0.7832167832167832,
 'auc': 0.8907979407979407}

In [65]:
evaluar_features(df_exploratorio, ["Title_agrupado_Rare", 
                                   "Title_agrupado_Mujer",
                                    "Title_agrupado_Niño",   
                                    "Sex_male","Pclass",
                                    #"Embarked_Q", "Embarked_S",
                                    "FamilySize_agrupado_Pequeña",
                                    "FamilySize_agrupado_Grande",
                                    "Age_bin_Joven",
                                    "Age_bin_Niño",
                                    "Age_bin_Adolescente",
                                    "Age_bin_Mayor"])

{'features': ['Title_agrupado_Rare',
  'Title_agrupado_Mujer',
  'Title_agrupado_Niño',
  'Sex_male',
  'Pclass',
  'FamilySize_agrupado_Pequeña',
  'FamilySize_agrupado_Grande',
  'Age_bin_Joven',
  'Age_bin_Niño',
  'Age_bin_Adolescente',
  'Age_bin_Mayor'],
 'f1': 0.7832167832167832,
 'auc': 0.8968468468468468}

In [66]:
evaluar_features(df_exploratorio, ["Title_agrupado_Rare", 
                                   "Title_agrupado_Mujer",
                                    "Title_agrupado_Niño",   
                                    "Sex_male","Pclass",
                                    "Embarked_Q", "Embarked_S",
                                    "FamilySize_agrupado_Pequeña",
                                    "FamilySize_agrupado_Grande",
                                    "Age"])

{'features': ['Title_agrupado_Rare',
  'Title_agrupado_Mujer',
  'Title_agrupado_Niño',
  'Sex_male',
  'Pclass',
  'Embarked_Q',
  'Embarked_S',
  'FamilySize_agrupado_Pequeña',
  'FamilySize_agrupado_Grande',
  'Age'],
 'f1': 0.7916666666666666,
 'auc': 0.8944658944658944}

In [67]:
df_exploratorio["Age_scaled"] = StandardScaler().fit_transform(df_exploratorio[["Age"]])

In [68]:
evaluar_features(df_exploratorio, ["Title_agrupado_Rare", 
                                   "Title_agrupado_Mujer",
                                    "Title_agrupado_Niño",   
                                    "Sex_male","Pclass",
                                    "Embarked_Q", "Embarked_S",
                                    "FamilySize_agrupado_Pequeña",
                                    "FamilySize_agrupado_Grande",
                                    "Age_scaled"])

{'features': ['Title_agrupado_Rare',
  'Title_agrupado_Mujer',
  'Title_agrupado_Niño',
  'Sex_male',
  'Pclass',
  'Embarked_Q',
  'Embarked_S',
  'FamilySize_agrupado_Pequeña',
  'FamilySize_agrupado_Grande',
  'Age_scaled'],
 'f1': 0.7916666666666666,
 'auc': 0.8944658944658944}

In [69]:

def evaluar_cv_threshold(df, features):
    
    X = df[features]
    y = df["Survived"]
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    f1_scores = []
    auc_scores = []
    best_thresholds = []
    
    for train_idx, val_idx in skf.split(X, y):
        
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        modelo = LogisticRegression(max_iter=1000)
        modelo.fit(X_train, y_train)
        
        y_proba = modelo.predict_proba(X_val)[:,1]
        
        thresholds = np.linspace(0,1,100)
        f1_temp = []
        
        for t in thresholds:
            y_pred = (y_proba >= t).astype(int)
            f1_temp.append(f1_score(y_val, y_pred))
        
        best_t = thresholds[np.argmax(f1_temp)]
        best_thresholds.append(best_t)
        
        # usar el mejor threshold en ese fold
        y_pred_final = (y_proba >= best_t).astype(int)
        
        f1_scores.append(f1_score(y_val, y_pred_final))
        auc_scores.append(roc_auc_score(y_val, y_proba))
    
    return {
        "features": features,
        "f1_mean": np.mean(f1_scores),
        "f1_std": np.std(f1_scores),
        "auc_mean": np.mean(auc_scores),
        "threshold_mean": np.mean(best_thresholds)
    }

In [70]:
evaluar_cv_threshold(df_exploratorio, ["Title_agrupado_Rare", 
                                   "Title_agrupado_Mujer",
                                    "Title_agrupado_Niño",   
                                    "Sex_male","Pclass",
                                    "Embarked_Q", "Embarked_S",
                                    "FamilySize_agrupado_Pequeña",
                                    "FamilySize_agrupado_Grande",
                                    "Age_scaled"])

{'features': ['Title_agrupado_Rare',
  'Title_agrupado_Mujer',
  'Title_agrupado_Niño',
  'Sex_male',
  'Pclass',
  'Embarked_Q',
  'Embarked_S',
  'FamilySize_agrupado_Pequeña',
  'FamilySize_agrupado_Grande',
  'Age_scaled'],
 'f1_mean': 0.7918741432277037,
 'f1_std': 0.01597923784033658,
 'auc_mean': 0.8711735536220507,
 'threshold_mean': 0.4282828282828284}

In [71]:
evaluar_cv_threshold(df_exploratorio, ["Title_agrupado_Rare", 
                                   "Title_agrupado_Mujer",
                                    "Title_agrupado_Niño",   
                                    "Sex_male","Pclass",
                                    "Embarked_Q", "Embarked_S",
                                    "FamilySize_agrupado_Pequeña",
                                    "FamilySize_agrupado_Grande",
                                    "Age"])

{'features': ['Title_agrupado_Rare',
  'Title_agrupado_Mujer',
  'Title_agrupado_Niño',
  'Sex_male',
  'Pclass',
  'Embarked_Q',
  'Embarked_S',
  'FamilySize_agrupado_Pequeña',
  'FamilySize_agrupado_Grande',
  'Age'],
 'f1_mean': 0.7906621580408559,
 'f1_std': 0.014572381150817132,
 'auc_mean': 0.871253476006931,
 'threshold_mean': 0.4242424242424243}

In [72]:
evaluar_cv_threshold(df_exploratorio, ["Title_agrupado_Rare", 
                                   "Title_agrupado_Mujer",
                                    "Title_agrupado_Niño",   
                                    "Sex_male","Pclass",
                                    "Embarked_Q", "Embarked_S",
                                    "FamilySize_agrupado_Pequeña",
                                    "FamilySize_agrupado_Grande",
                                    "Age_bin_Joven",
                                    "Age_bin_Niño",
                                    "Age_bin_Adolescente",
                                    "Age_bin_Mayor"])

{'features': ['Title_agrupado_Rare',
  'Title_agrupado_Mujer',
  'Title_agrupado_Niño',
  'Sex_male',
  'Pclass',
  'Embarked_Q',
  'Embarked_S',
  'FamilySize_agrupado_Pequeña',
  'FamilySize_agrupado_Grande',
  'Age_bin_Joven',
  'Age_bin_Niño',
  'Age_bin_Adolescente',
  'Age_bin_Mayor'],
 'f1_mean': 0.7876768216213714,
 'f1_std': 0.015479512832600295,
 'auc_mean': 0.870650540340878,
 'threshold_mean': 0.44040404040404046}

In [73]:
evaluar_cv_threshold(df_exploratorio, ["Title_agrupado_Rare", 
                                   "Title_agrupado_Mujer",
                                    "Title_agrupado_Niño",   
                                    "Sex_male","Pclass",
                                    "Embarked_Q", "Embarked_S",
                                    "FamilySize_agrupado_Pequeña",
                                    "FamilySize_agrupado_Grande"])

{'features': ['Title_agrupado_Rare',
  'Title_agrupado_Mujer',
  'Title_agrupado_Niño',
  'Sex_male',
  'Pclass',
  'Embarked_Q',
  'Embarked_S',
  'FamilySize_agrupado_Pequeña',
  'FamilySize_agrupado_Grande'],
 'f1_mean': 0.7814581166354622,
 'f1_std': 0.01031803156322642,
 'auc_mean': 0.8686420944705983,
 'threshold_mean': 0.3939393939393939}

In [74]:
evaluar_cv_threshold(df_exploratorio, ["Title_agrupado_Rare", 
                                   "Title_agrupado_Mujer",
                                    "Title_agrupado_Niño",   
                                    "Sex_male","Pclass","Age",
                                    "FamilySize_agrupado_Pequeña",
                                    "FamilySize_agrupado_Grande"])

{'features': ['Title_agrupado_Rare',
  'Title_agrupado_Mujer',
  'Title_agrupado_Niño',
  'Sex_male',
  'Pclass',
  'Age',
  'FamilySize_agrupado_Pequeña',
  'FamilySize_agrupado_Grande'],
 'f1_mean': 0.7917084242106421,
 'f1_std': 0.018040093651025442,
 'auc_mean': 0.8695489794351218,
 'threshold_mean': 0.4}